# 7. Version Control and Collaboration with GitHub

Every materials scientist eventually meets `analysis_v2_final.ipynb`, then
`analysis_v2_final_REALLY.ipynb`, then a version emailed by a labmate that
conflicts with both. **Version control** — specifically **git**, and
**GitHub**, the website built around it — solves this properly: instead of
duplicating whole files by hand, it tracks the exact *changes* between
versions, lets you jump back to any earlier state on demand, and lets
several people edit the same project without overwriting each other. This is
the same underlying discipline as Part II's FAIR data principles, applied
to *code* instead of *data*: a well-kept git history is itself a record of
exactly how an analysis came to look the way it does — which is precisely
what the course project's grading rubric rewards under "Reproducibility."

This notebook builds a small **simulated** version-control system in plain
Python first, so the underlying ideas (commit, history, branch) are visible
and safely runnable with no setup at all — then shows the real `git`
commands that do the same thing on your own computer, and how GitHub adds
collaboration on top.

**Topics**
1. Why version control, not just careful file naming?
2. A commit, simulated: snapshots you can always return to
3. The real thing: `git init`, `add`, `commit`, `log`
4. Branches: working on an idea without breaking what works
5. GitHub: remotes, `clone`, `push`, `pull`
6. Collaborating: forks, pull requests, and merge conflicts
7. Practical hygiene: `.gitignore` and notebooks in git

## 7.1 Why Version Control?

Renaming files by hand (`_v2`, `_final`, `_final_final`) fails for reasons
that get worse the longer a project runs: there is no record of *why*
something changed, no way to see exactly what changed between two versions
without eyeballing both files side by side, and no sane way for two people
to work on the same file at once without one of them overwriting the
other's work. Version control fixes all three: every saved change (a
**commit**) records exactly what changed, who changed it, and why (in a
short message you write yourself); the complete history is always
available; and — as Section 7.4 and 7.6 show — two people's independent
changes can usually be combined automatically.

## 7.2 A Commit, Simulated

Before touching real `git`, it helps to see the core idea with nothing but
a Python list — a **commit** is just a labelled snapshot of a project's
files at one point in time, and a project's **history** is a chain of these
snapshots, each one remembering which snapshot came directly before it.
Real git's actual internals are more sophisticated (content-addressed
storage, not full-file copies), but the *mental model* below — a chain of
labelled, timestamped snapshots — is exactly the right one for everything
that follows.

In [1]:
import hashlib
from datetime import datetime, timedelta

class ToyRepo:
    """A tiny, in-memory stand-in for git's commit history."""
    def __init__(self):
        self.commits = []          # chain of snapshots, oldest first
        self.working_files = {}    # current file contents

    def _commit_id(self, message, timestamp):
        raw = f'{message}{timestamp}{len(self.commits)}'.encode()
        return hashlib.sha1(raw).hexdigest()[:7]   # git shows short hashes like this

    def commit(self, message, author='you'):
        parent = self.commits[-1]['id'] if self.commits else None
        timestamp = datetime.now() + timedelta(seconds=len(self.commits))
        snapshot = {
            'id': self._commit_id(message, timestamp),
            'parent': parent,
            'message': message,
            'author': author,
            'timestamp': timestamp,
            'files': dict(self.working_files),   # a full snapshot, copied
        }
        self.commits.append(snapshot)
        return snapshot['id']

    def log(self):
        for c in reversed(self.commits):
            print(f"{c['id']}  {c['timestamp']:%Y-%m-%d %H:%M:%S}  {c['author']:<8} {c['message']}")


repo = ToyRepo()
repo.working_files['analysis.py'] = "print('load data')"
first_commit = repo.commit('Initial analysis script')

repo.working_files['analysis.py'] = "print('load data')\nprint('clean data')"
repo.commit('Add data cleaning step')

repo.working_files['analysis.py'] += "\nprint('fit model')"
repo.working_files['README.md'] = '# Battery capacity analysis'
repo.commit('Add model fitting and a README')

repo.log()


730a738  2026-08-13 13:36:33  you      Add model fitting and a README
6b0f1de  2026-08-13 13:36:32  you      Add data cleaning step
7ed6bec  2026-08-13 13:36:31  you      Initial analysis script


Three commits, each a complete snapshot, each remembering its `parent` —
that parent link is what turns a pile of snapshots into a proper
**history**, and it is exactly what real `git log` prints (with real hashes
in place of `ToyRepo`'s short simulated ones). Nothing here is ever
overwritten: an earlier snapshot is always still reachable through the
chain, which is *why* version control lets you recover a working version
after a bad change — a real, common lifesaver `mv old_file backup_file`
never quite manages.

## 7.3 The Real Thing: `git init`, `add`, `commit`, `log`

Real `git` on your own computer follows exactly the pattern above, as
commands typed in a terminal inside your project folder (install it from
[git-scm.com](https://git-scm.com) if `git --version` fails). These are
illustrative — copy them into your own terminal, not into this notebook,
since this book's own repository is not meant to be `git init`-ed from
inside itself.

```bash
git init                          # start tracking this folder
git add analysis.py               # stage a file: "include this in the next commit"
git commit -m "Initial analysis script"

# ... edit analysis.py ...
git add analysis.py
git commit -m "Add data cleaning step"

git log                           # the real equivalent of ToyRepo.log() above
```

`git add` has no equivalent in `ToyRepo` above — real git distinguishes
*changed on disk* from *staged for the next commit*, so you can build up a
commit from several edited files one at a time, deciding exactly what goes
in together (`git add file1.py`, leave `file2.py` for a separate commit).
`git status` at any point tells you what is changed, staged, or untracked —
the single most useful command for "wait, what state is this in?".

## 7.4 Branches: Trying an Idea Without Breaking What Works

So far, history has been a single, straight chain. A **branch** lets that
chain fork: you keep a stable line of commits (conventionally named `main`)
untouched, while trying something risky — a new feature, an experimental
analysis method — on a separate line that starts from the same point.
Simulating this needs only one small addition to `ToyRepo`: instead of one
list of commits, track *named pointers* to specific commits, and let
`commit()` advance whichever branch is currently checked out.

In [2]:
class BranchingRepo(ToyRepo):
    def __init__(self):
        super().__init__()
        self.branches = {'main': None}   # branch name -> commit id it points to
        self.current_branch = 'main'

    def commit(self, message, author='you'):
        commit_id = super().commit(message, author)
        self.branches[self.current_branch] = commit_id
        return commit_id

    def branch(self, name):
        self.branches[name] = self.branches[self.current_branch]

    def checkout(self, name):
        self.current_branch = name

    def status(self):
        print(f"On branch '{self.current_branch}'")
        for name, cid in self.branches.items():
            marker = ' <-- you are here' if name == self.current_branch else ''
            print(f"  {name}: {cid}{marker}")


brepo = BranchingRepo()
brepo.working_files['model.py'] = "model = 'MLR'"
brepo.commit('Baseline: linear regression model')

brepo.branch('try-random-forest')     # fork off from the current commit
brepo.checkout('try-random-forest')
brepo.working_files['model.py'] = "model = 'RandomForest'"
brepo.commit('Experiment: swap in a Random Forest')

brepo.status()


On branch 'try-random-forest'
  main: 45cfcfe
  try-random-forest: 3140751 <-- you are here


`main` still points at the safe, working baseline commit; the experiment
lives entirely on `try-random-forest` and cannot break `main` no matter how
badly it goes. If the Random Forest experiment works out, its commits get
**merged** back into `main` (`git merge try-random-forest`, or a GitHub pull
request, Section 7.6); if it does not, you simply stop using that branch —
`main` was never touched. This is exactly the workflow behind
Notebook 6's checkpoint suggestion of testing a risky package upgrade in a
throwaway conda environment before touching your real one — isolate the
risk, keep a safe fallback, decide once you know the outcome.

## 7.5 GitHub: Remotes, `clone`, `push`, `pull`

Everything so far lives only on one computer. **GitHub** hosts a copy of a
git repository on a server, so it can act as a shared meeting point: a
**remote**. The vocabulary maps directly onto what you already know —
`clone` is "download a full copy of someone's repository, history and all,"
`push` is "send my new local commits up to GitHub," `pull` is "download
commits someone else pushed, and merge them into mine."

```bash
git clone https://github.com/some-org/some-repo.git   # get a full copy, once
cd some-repo

# ... make changes, then commit them locally as in Section 7.3 ...
git push                          # send your new commits to GitHub
git pull                          # fetch and merge anyone else's new commits
```

A repository with no remote at all (Section 7.3) is still completely valid
git — useful even solo, purely for the history and branching benefits. A
remote is what turns that same tool into a collaboration platform, and
GitHub specifically (rather than git in general) is what adds the website:
browsing a project's files and history in a browser, issue trackers, and —
the collaboration mechanism Section 7.6 covers — pull requests.

## 7.6 Collaborating: Forks, Pull Requests, and Merge Conflicts

Two common collaboration shapes on GitHub:

- **Direct collaborator**: someone adds you to their repository; you `clone`
  it, create a branch (Section 7.4) for your change, `push` that branch, then
  open a **pull request** — a formal proposal, visible on GitHub, to merge
  your branch into `main`, which the project owner (or a labmate) can review,
  comment on, and approve before it actually merges.
- **Fork**: for a project you do not have write access to, GitHub's "Fork"
  button makes your own personal copy on GitHub first; you `clone` *that*,
  make changes, `push` to your fork, then open a pull request from your fork
  back to the original — the standard way open-source contributions (and
  most student contributions to a supervisor's repository) work.

**Merge conflicts** happen when two branches change the *same lines* of the
same file differently — git cannot guess which version you want, so it
stops and asks you to decide by hand, marking both versions directly in the
file:

```
<<<<<<< HEAD
model = 'RandomForest'
=======
model = 'GradientBoosting'
>>>>>>> try-random-forest
```

You edit the file to keep whichever version (or a hand-merged combination)
is correct, delete the `<<<<<<<`/`=======`/`>>>>>>>` markers, then commit —
this is the one part of git that occasionally needs a human, precisely
because it means two people had a genuinely different idea about the same
line of code, which git cannot resolve for you.

## 7.7 Practical Hygiene: `.gitignore` and Notebooks in Git

Two habits worth adopting from your very first repository:

**`.gitignore`** — a plain-text file listing patterns git should never
track: build artefacts, large data files, and anything that regenerates
automatically. This course's own repository excludes exactly the kind of
things a data-analysis project accumulates:

```
_build/
*.pyc
__pycache__/
.ipynb_checkpoints/
*.csv
```

Large raw data files in particular belong in a proper data store (Notebook
3 or 4's databases, or a dedicated data-versioning tool), not committed
directly into git — git keeps *every* version of every tracked file
forever, so a repeatedly-overwritten multi-GB CSV bloats the repository
permanently.

**Notebooks specifically** are trickier to track well than plain `.py`
files: a `.ipynb` file stores cell *outputs* — including embedded PNG
images, base64-encoded — right alongside the code, so re-running a notebook
and re-saving it changes the file even when no code actually changed,
producing noisy, hard-to-read diffs. Two common fixes: clear all outputs
before committing (`jupyter nbconvert --clear-output`), or use a tool like
`nbstripout` that does this automatically on every commit — worth setting
up before your course-project notebook's git history turns into hundreds of
near-identical "re-ran cell 47" commits.

**Exercise 1**: Using `BranchingRepo` from Section 7.4, create a second
branch called `try-gbm` from the same starting commit as
`try-random-forest`, commit a change to `model.py` on it, then print
`.status()`. Confirm all three branches (`main`, `try-random-forest`,
`try-gbm`) point at three different commits, and that `main` still points
at the original baseline.

**Exercise 2**: Extend `ToyRepo.log()` (or write a new method) so it can
also print the history of just *one file*, by walking the commit chain and
only printing a commit if that file's contents differ from its parent's
version — this is a simplified version of what `git log -- <filename>` does
for real.

**Exercise 3**: If you have `git` installed, create a real, throwaway
repository somewhere *outside* this course's own folder (e.g. a new empty
directory), and reproduce Sections 7.3–7.4 for real: `git init`, two
commits, a branch, and `git log --oneline --graph --all` to see the
branching structure git itself draws in the terminal. Compare it to
`BranchingRepo.status()`'s output above.

**Exercise 4**: Find a small, active open-source project on GitHub in a
field you are interested in. Without making any changes, browse its commit
history and its (open or closed) pull requests. Find one pull request that
was merged and read the discussion — what did a reviewer ask the
contributor to change before merging?